# ERGT 02 - Adaptive Competitive Alpha

This notebook is a separate follow-up to `ERGT_01_Attention_Evidence_Ladder.ipynb`. It does not modify the fixed-alpha evidence record. The goal here is to test a competitive alpha policy: geometry is allowed to grow when rolling validation-loss slope and stability evidence support it, while rigidity risks only penalize growth instead of acting as a hard `geo/qk` ceiling.

Default profile is `adaptive_smoke`. For the guarded 2000-step run, set `RUN_PROFILE = "adaptive_2000_competitive"` in the setup cell.

In [ ]:
from __future__ import annotations

import csv
import datetime as dt
import json
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
PROJECT_ROOT = Path("/content/ERGT") if Path("/content").exists() else Path.cwd()

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False

if running_in_colab():
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import torch
    DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEFAULT_DEVICE = "cpu"

RUN_PROFILE = "adaptive_smoke"  # "adaptive_smoke" or "adaptive_2000_competitive"
DEVICE = DEFAULT_DEVICE
SEED = 2027
RUN_UNIT_TESTS = True
RUN_TRAINING = True
PREPARE_DATA_IF_MISSING = running_in_colab()
INCLUDE_INSTANTANEOUS_CONTROL = True
AUTO_SHUTDOWN_COLAB_RUNTIME = True
AUTO_SHUTDOWN_ON_FAILURE = True
AUTO_SHUTDOWN_DELAY_SECONDS = 30

def run_cmd(cmd, *, timeout_sec=None, check=True):
    start = time.perf_counter()
    print("\n$ " + " ".join(str(part) for part in cmd), flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        env=env,
    )
    lines = []
    line_queue = queue.Queue()

    def _reader():
        assert process.stdout is not None
        for line in process.stdout:
            line_queue.put(line)

    reader = threading.Thread(target=_reader, daemon=True)
    reader.start()
    timed_out = False
    while process.poll() is None:
        try:
            line = line_queue.get(timeout=0.2)
        except queue.Empty:
            line = None
        if line is not None:
            print(line, end="", flush=True)
            lines.append(line)
        if timeout_sec is not None and time.perf_counter() - start > timeout_sec:
            timed_out = True
            process.kill()
            break
    reader.join(timeout=1.0)
    while not line_queue.empty():
        line = line_queue.get_nowait()
        print(line, end="", flush=True)
        lines.append(line)
    elapsed = time.perf_counter() - start
    returncode = "timeout" if timed_out else process.returncode
    print(f"returncode={returncode}, elapsed={elapsed / 60:.2f} min", flush=True)
    if timed_out:
        raise TimeoutError(f"Command timed out after {timeout_sec} seconds: {cmd}")
    if check and process.returncode != 0:
        raise RuntimeError(f"Command failed with code {process.returncode}: {cmd}")
    return {"returncode": process.returncode, "elapsed_seconds": elapsed, "output": "".join(lines)}

def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def shutdown_colab_runtime(reason: str):
    if not running_in_colab() or not AUTO_SHUTDOWN_COLAB_RUNTIME:
        return
    print(f"Auto-shutdown requested: {reason}. Waiting {AUTO_SHUTDOWN_DELAY_SECONDS}s.", flush=True)
    time.sleep(AUTO_SHUTDOWN_DELAY_SECONDS)
    try:
        from google.colab import runtime  # type: ignore
        runtime.unassign()
    except Exception:
        os.kill(os.getpid(), 9)

print({"project_root": str(PROJECT_ROOT), "run_profile": RUN_PROFILE, "device": DEVICE})

In [ ]:
PROFILE_SETTINGS = {
    "adaptive_smoke": {
        "context_length": 64,
        "max_steps": 40,
        "eval_interval": 10,
        "max_eval_batches": 2,
        "batch_size": 4,
        "n_layers": 2,
        "n_heads": 4,
        "hidden_dim": 128,
        "ffn_dim": 512,
        "warmup_steps": 4,
        "per_run_timeout_minutes": 10,
        "conditions": ["baseline", "alpha_zero", "real_memory_d_adaptive", "random_memory_d_adaptive", "shuffled_memory_d_adaptive"],
    },
    "adaptive_2000_competitive": {
        "context_length": 256,
        "max_steps": 2000,
        "eval_interval": 100,
        "max_eval_batches": None,
        "batch_size": 16,
        "n_layers": 6,
        "n_heads": 8,
        "hidden_dim": 512,
        "ffn_dim": 2048,
        "warmup_steps": 200,
        "per_run_timeout_minutes": 180,
        "conditions": ["baseline", "alpha_zero", "real_memory_d_adaptive", "random_memory_d_adaptive", "shuffled_memory_d_adaptive", "no_memory_real_d_adaptive", "instantaneous_real_d_adaptive"],
    },
}

if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

BUDGET = PROFILE_SETTINGS[RUN_PROFILE]
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = PROJECT_ROOT / "runs" / "notebook_02_adaptive_competitive_alpha" / f"{timestamp}_{RUN_PROFILE}"
CONFIG_DIR = RUN_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / f"wikitext2_gpt2_ctx{BUDGET['context_length']}"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "metadata.json").exists():
    if not PREPARE_DATA_IF_MISSING:
        raise FileNotFoundError(f"Prepared data missing: {DATA_DIR}")
    run_cmd([
        sys.executable,
        "-u",
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--context-length",
        BUDGET["context_length"],
    ], timeout_sec=60 * 60)

if RUN_UNIT_TESTS:
    run_cmd([
        sys.executable,
        "-m",
        "pytest",
        "tests/test_adaptive_alpha.py",
        "tests/test_geo_attention.py",
        "tests/test_ergt_v1_training.py",
    ], timeout_sec=15 * 60)

print({"run_root": str(RUN_ROOT), "data_dir": str(DATA_DIR)})

In [ ]:
def common_dataset():
    return {
        "name": "wikitext-2",
        "split": "standard",
        "tokenizer": "gpt2",
        "context_length": BUDGET["context_length"],
    }

def common_model(model_type: str):
    return {
        "type": model_type,
        "vocab_size": None,
        "n_layers": BUDGET["n_layers"],
        "n_heads": BUDGET["n_heads"],
        "hidden_dim": BUDGET["hidden_dim"],
        "ffn_dim": BUDGET["ffn_dim"],
        "dropout": 0.1,
        "bias": True,
        "positional_encoding": "learned_absolute",
    }

def common_training():
    return {
        "optimizer": "adamw",
        "learning_rate": 0.0003,
        "betas": [0.9, 0.95],
        "weight_decay": 0.1,
        "batch_size": BUDGET["batch_size"],
        "max_steps": BUDGET["max_steps"],
        "warmup_steps": BUDGET["warmup_steps"],
        "grad_clip": 1.0,
        "eval_interval": BUDGET["eval_interval"],
        "checkpoint_interval": BUDGET["max_steps"],
        "max_eval_batches": BUDGET["max_eval_batches"],
    }

def common_logging():
    return {
        "train_log": "train_log.jsonl",
        "progress_log": "progress_log.jsonl",
        "adaptive_alpha_log": "adaptive_alpha_log.jsonl",
        "results": "metrics.json",
        "model_summary": "model_summary.json",
        "log_geometry_diagnostics": True,
    }

def baseline_config(condition: str):
    return {
        "run": {"phase": "adaptive_competitive_alpha", "condition": condition, "seed": SEED, "output_dir": str(RUN_ROOT / condition)},
        "dataset": common_dataset(),
        "model": common_model("transformer_baseline"),
        "training": common_training(),
        "logging": common_logging(),
    }

def ergt_config(condition: str, distance_mode: str, alpha: float, adaptive: bool):
    attention = {
        "type": "geo_attention",
        "distance_mode": distance_mode,
        "head_sharing": "shared_d",
        "alpha": {"mode": "fixed", "initial_value": alpha, "non_negative": True, "warmup_steps": 0},
        "gradient_mode": "detached_d",
        "causal_runtime_distance": True,
        "max_causal_step": 1,
        "memory": {"decay": 0.7, "eta": 0.3, "gate_floor": 0.05, "min_context_edges": 2},
    }
    config = {
        "run": {"phase": "adaptive_competitive_alpha", "condition": condition, "seed": SEED, "output_dir": str(RUN_ROOT / condition)},
        "dataset": common_dataset(),
        "model": common_model("ergt_v1"),
        "attention": attention,
        "relational_graph": {"kernel": "sigmoid_cosine", "graph_heads": 1, "normalize_hidden": True, "diagonal_policy": "keep_for_distance"},
        "distance": {"formula": "-log(W + epsilon)", "epsilon": 1e-6, "normalization": "offdiag_zscore_clamp", "clip_value": 5.0, "diagonal_policy": "zero"},
        "training": common_training(),
        "logging": common_logging(),
    }
    if adaptive:
        config["adaptive_alpha"] = {
            "initial_alpha": 0.0,
            "min_alpha": 0.0,
            "max_alpha": 0.25,
            "decision_interval_steps": BUDGET["eval_interval"],
            "min_points_for_slope": 4,
            "slope_window_points": 5,
            "exploration_points": 4,
            "exploration_alpha": 0.025,
            "exploration_step": 0.04,
            "growth_step": 0.04,
            "decay_step": 0.02,
            "max_change_per_decision": 0.01,
            "inertia": 0.8,
            "score_scale": 0.001,
            "positive_margin": 0.0002,
            "negative_margin": -0.0002,
            "slope_gain_weight": 1.0,
            "advantage_weight": 0.25,
            "geo_qk_target": 0.06,
            "geo_qk_risk_weight": 0.0004,
            "entropy_drop_weight": 0.0002,
            "max_probability_target": 0.35,
            "max_probability_risk_weight": 0.0003,
            "ema_beta": 0.7,
        }
    return config

condition_specs = {
    "baseline": {"script": "experiments/train_baseline.py", "config": baseline_config("baseline"), "adaptive": False},
    "alpha_zero": {"script": "experiments/train_ergt_v1.py", "config": ergt_config("alpha_zero", "zero_d", 0.0, False), "adaptive": False},
    "real_memory_d_adaptive": {"script": "experiments/train_ergt_adaptive_alpha.py", "config": ergt_config("real_memory_d_adaptive", "real_stable_causal_d", 0.0, True), "adaptive": True},
    "random_memory_d_adaptive": {"script": "experiments/train_ergt_adaptive_alpha.py", "config": ergt_config("random_memory_d_adaptive", "random_stable_causal_d", 0.0, True), "adaptive": True},
    "shuffled_memory_d_adaptive": {"script": "experiments/train_ergt_adaptive_alpha.py", "config": ergt_config("shuffled_memory_d_adaptive", "shuffled_stable_causal_d", 0.0, True), "adaptive": True},
    "no_memory_real_d_adaptive": {"script": "experiments/train_ergt_adaptive_alpha.py", "config": ergt_config("no_memory_real_d_adaptive", "no_memory_real_d", 0.0, True), "adaptive": True},
    "instantaneous_real_d_adaptive": {"script": "experiments/train_ergt_adaptive_alpha.py", "config": ergt_config("instantaneous_real_d_adaptive", "instantaneous_real_d", 0.0, True), "adaptive": True},
}

run_plan = []
for condition in BUDGET["conditions"]:
    spec = condition_specs[condition]
    config_path = CONFIG_DIR / f"{condition}.json"
    write_json(config_path, spec["config"])
    run_plan.append({"condition": condition, "script": spec["script"], "config_path": config_path, "adaptive": spec["adaptive"]})

print(json.dumps({"run_plan": [item["condition"] for item in run_plan]}, indent=2))

In [ ]:
training_records = []
per_run_timeout_sec = int(BUDGET["per_run_timeout_minutes"] * 60)
baseline_progress = RUN_ROOT / "baseline" / "progress_log.jsonl"

try:
    if RUN_TRAINING:
        for item in run_plan:
            cmd = [sys.executable, "-u", item["script"], "--config", item["config_path"], "--device", DEVICE]
            if item["adaptive"]:
                if not baseline_progress.exists():
                    raise FileNotFoundError(f"Missing baseline progress reference: {baseline_progress}")
                cmd.extend(["--reference-progress", baseline_progress])
            result = run_cmd(cmd, timeout_sec=per_run_timeout_sec)
            training_records.append({"condition": item["condition"], **result})
    write_json(RUN_ROOT / "training_records.json", training_records)
except Exception as exc:
    write_json(RUN_ROOT / "training_failure.json", {"error": repr(exc), "records": training_records})
    if AUTO_SHUTDOWN_ON_FAILURE:
        shutdown_colab_runtime("failure")
    raise

In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

rows = []
baseline_rows = {row["step"]: row for row in read_jsonl(baseline_progress)}
for item in run_plan:
    progress_path = RUN_ROOT / item["condition"] / "progress_log.jsonl"
    for row in read_jsonl(progress_path):
        base = baseline_rows.get(row["step"], {})
        baseline_loss = base.get("validation_loss")
        val_loss = row.get("validation_loss")
        rows.append({
            "step": row.get("step"),
            "condition": item["condition"],
            "validation_loss": val_loss,
            "baseline_validation_loss": baseline_loss,
            "baseline_centered_improvement": None if baseline_loss is None or val_loss is None else baseline_loss - val_loss,
            "alpha_effective": row.get("alpha_effective"),
            "alpha_next": row.get("alpha_next"),
            "adaptive_score": row.get("adaptive_score"),
            "alpha_decision": row.get("alpha_decision"),
            "geo_to_qk_ratio": row.get("geo_to_qk_ratio"),
            "attention_entropy": row.get("attention_entropy"),
            "mean_max_probability": row.get("mean_max_probability"),
            "elapsed_minutes": row.get("elapsed_minutes"),
        })

summary_csv = RUN_ROOT / "adaptive_competitive_stepwise.csv"
fields = ["step", "condition", "validation_loss", "baseline_validation_loss", "baseline_centered_improvement", "alpha_effective", "alpha_next", "adaptive_score", "alpha_decision", "geo_to_qk_ratio", "attention_entropy", "mean_max_probability", "elapsed_minutes"]
with summary_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for row in sorted(rows, key=lambda item: (item["step"] or -1, item["condition"])):
        writer.writerow(row)

print(f"Wrote {summary_csv}")
for row in rows[-20:]:
    print(row)

try:
    import matplotlib.pyplot as plt
    plot_path = RUN_ROOT / "adaptive_alpha_trajectory.png"
    fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
    for condition in sorted({row["condition"] for row in rows}):
        condition_rows = sorted([row for row in rows if row["condition"] == condition], key=lambda row: row["step"])
        steps = [row["step"] for row in condition_rows]
        axes[0].plot(steps, [row["validation_loss"] for row in condition_rows], label=condition)
        axes[1].plot(steps, [row["baseline_centered_improvement"] for row in condition_rows], label=condition)
        axes[2].plot(steps, [row["alpha_next"] for row in condition_rows], label=condition)
    axes[0].set_ylabel("val loss")
    axes[1].set_ylabel("baseline - condition")
    axes[2].set_ylabel("alpha next")
    axes[2].set_xlabel("step")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(plot_path, dpi=140)
    print(f"Wrote {plot_path}")
except Exception as exc:
    print(f"Plot skipped: {exc}")

In [ ]:
bundle_name = "ergt_02_adaptive_competitive_alpha_bundle.zip"
bundle_path = RUN_ROOT / bundle_name
include_suffixes = {".json", ".jsonl", ".csv", ".png", ".txt", ".md"}
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in RUN_ROOT.rglob("*"):
        if path.is_dir() or "checkpoints" in path.parts:
            continue
        if path.suffix.lower() in include_suffixes:
            zf.write(path, path.relative_to(RUN_ROOT))

download_path = Path("/content") / bundle_name if running_in_colab() else bundle_path
if running_in_colab():
    shutil.copyfile(bundle_path, download_path)

print("Fixed output bundle name:")
print(bundle_name)
print("Default local review path after Colab download:")
print("C:\\Users\\Administrator\\Downloads\\" + bundle_name)
print(f"Bundle path in runtime: {download_path}")

shutdown_colab_runtime("success")